In [ ]:
import numpy as np

# --- 1. Calculate Movie Popularity (based on 'ratings') ---
# Calculate average rating and number of ratings for each movie
movie_stats = ratings.groupby('tmdbId')['rating'].agg(['mean', 'count']).reset_index()
movie_stats.columns = ['tmdbId', 'average_rating', 'num_ratings']

# Merge with movie titles for easier interpretation and display
popular_movies = movie_stats.merge(movies[['id', 'title']], left_on='tmdbId', right_on='id', how='inner').drop(columns=['id'])

# Calculate m (minimum votes required, e.g., 90th percentile)
m_ratings = popular_movies['num_ratings'].quantile(0.90)

# Define the popularity function using simple average rating
def simple_popularity_rating(x, m=m_ratings):
    v = x['num_ratings']
    R = x['average_rating']
    if v < m:
        return 0 # Movies with fewer than 'm' ratings get a low score
    return R

# Apply the popularity function to calculate popularity score
popular_movies['popularity_score'] = popular_movies.apply(simple_popularity_rating, axis=1)

# Sort movies by their popularity score in descending order
popular_movies = popular_movies.sort_values(by='popularity_score', ascending=False)

print("### 1. Movie Popularity Calculation (based on ratings):")
print(f"Using average rating for popularity. Movies with fewer than {int(m_ratings)} ratings (90th percentile) get a low score.")
print("Top 5 most popular movies:")
display(popular_movies[['title', 'average_rating', 'num_ratings', 'popularity_score']].head())

def recommend_popular_movies(user_id, num_recommendations=10, watched_movie_ids=None, popular_df=popular_movies):
    if watched_movie_ids is None:
        watched_movie_ids = set()
    unwatched_recommendations = popular_df[~popular_df['tmdbId'].isin(watched_movie_ids)]
    return unwatched_recommendations.head(num_recommendations)['tmdbId'].tolist()


### 1. Movie Popularity Calculation (based on ratings):
Using average rating for popularity. Movies with fewer than 1291 ratings (90th percentile) get a low score.
Top 5 most popular movies:


,title,average_rating,num_ratings,popularity_score
223,The Shawshank Redemption,4.427742,90045,4.427742
186,The Godfather,4.339794,56830,4.339794
485,The Usual Suspects,4.300198,59131,4.300198
321,Schindler's List,4.266668,67376,4.266668
188,The Godfather: Part II,4.263114,36621,4.263114


In [ ]:
# --- 2. Calculate TMDB Popularity ---
# Filter for qualified movies based on TMDB's vote_count and vote_average
m_tmdb = movies['vote_count'].quantile(0.90)
qualified_movies = movies[(movies['vote_count'] >= m_tmdb) & (movies['vote_count'].notna()) & (movies['vote_average'].notna())].copy()
tmdb_popular_movies = qualified_movies.sort_values(by='vote_average', ascending=False)

print("### 2. TMDB Popularity Calculation:")
print(f"Filtered for movies with vote_count >= {int(m_tmdb)} (90th percentile) and valid vote_average.")
print("Top 5 most popular movies from TMDB:")
display(tmdb_popular_movies[['title', 'vote_average', 'vote_count']].head())

def recommend_tmdb_popular_movies(user_id, num_recommendations=10, watched_movie_ids=None, popular_df=tmdb_popular_movies):
    if watched_movie_ids is None:
        watched_movie_ids = set()
    unwatched_recommendations = popular_df[~popular_df['id'].isin(watched_movie_ids)]
    return unwatched_recommendations.head(num_recommendations)['id'].tolist()

### 2. TMDB Popularity Calculation:
Filtered for movies with vote_count >= 160 (90th percentile) and valid vote_average.
Top 5 most popular movies from TMDB:


,title,vote_average,vote_count
10309,Dilwale Dulhania Le Jayenge,9.1,661.0
39085,Planet Earth,8.8,176.0
40251,Your Name.,8.5,1030.0
314,The Shawshank Redemption,8.5,8358.0
834,The Godfather,8.5,6024.0


In [ ]:
# --- 3. Prepare for Random Recommendations ---
# Eligible movies for random recommendations: e.g., movies with at least 3 ratings and average rating >= 3
# Using 'popular_movies' which already has 'average_rating' and 'num_ratings'
m_random_eligible = popular_movies['num_ratings'].quantile(0.50)
eligible_random_movie_ids = popular_movies[
    (popular_movies['average_rating'] >= 3) &
    (popular_movies['num_ratings'] >= m_random_eligible)
]['tmdbId'].tolist()

print("### 3. Eligible Movies for Random Recommendations:")
print(f"Number of eligible movies: {len(eligible_random_movie_ids):,}")
print(f"Example 5 random TMDB IDs: {np.random.choice(eligible_random_movie_ids, 5, replace=False).tolist()}")

def recommend_random_eligible_movies(user_id, num_recommendations=10, watched_movie_ids=None, eligible_movie_ids=None):
    if watched_movie_ids is None:
        watched_movie_ids = set()
    if eligible_movie_ids is None:
        eligible_movie_ids = []
    potential_recommendations = [mid for mid in eligible_movie_ids if mid not in watched_movie_ids]
    if len(potential_recommendations) <= num_recommendations:
        return potential_recommendations
    else:
        return np.random.choice(potential_recommendations, num_recommendations, replace=False).tolist()

### 3. Eligible Movies for Random Recommendations:
Number of eligible movies: 11,692
Example 5 random TMDB IDs: [179538, 13528, 170657, 55292, 9740]


## Serendipity Metric

Serendipity in recommender systems implies recommending items that are both **relevant** and **unexpected/novel** to the user. A simple way to quantify this is to combine relevance with an inverse measure of item popularity (as a proxy for unexpectedness).

We will define serendipity for a recommended item as:

$Serendipity_{item} = Relevance_{item} \times (1 - NormalizedPopularity_{item})$

Where:
*   $Relevance_{item}$ is 1 if the item is in the ground truth relevant set, and 0 otherwise.
*   $NormalizedPopularity_{item}$ is the `popularity_score` (average rating from our data) for the item, normalized to a range between 0 and 1 (by dividing by the maximum possible rating, which is 5.0). A higher normalized popularity means the item is more 'expected'.

The overall serendipity for a list of recommendations will be the average of the individual item serendipity scores.


In [ ]:
import numpy as np
import pandas as pd

def calculate_dcg(relevance_scores, k):
    dcg = 0.0
    for i in range(min(k, len(relevance_scores))):
        dcg += relevance_scores[i] / np.log2(i + 2)
    return dcg

def calculate_ndcg(recommended_items, ground_truth_relevant_items, k):
    # Create binary relevance scores for recommended items
    relevance_scores = [1 if item in ground_truth_relevant_items else 0 for item in recommended_items]

    # Calculate DCG for the recommended list
    dcg = calculate_dcg(relevance_scores, k)

    # Calculate IDCG (Ideal DCG) for the ground truth relevant items
    # First, create ideal relevance scores (all relevant items first)
    ideal_relevance_scores = [1] * len(ground_truth_relevant_items)
    idcg = calculate_dcg(ideal_relevance_scores, k)

    if idcg == 0:
        return 0.0  # Avoid division by zero if there are no relevant items
    return dcg / idcg

def calculate_precision_recall(recommended_items, ground_truth_relevant_items, k):
    recommended_set = set(recommended_items[:k])
    ground_truth_set = set(ground_truth_relevant_items)

    true_positives = len(recommended_set.intersection(ground_truth_set))

    precision = true_positives / k if k > 0 else 0
    recall = true_positives / len(ground_truth_set) if len(ground_truth_set) > 0 else 0

    return precision, recall

In [ ]:
def calculate_serendipity(recommended_items, ground_truth_relevant_items, item_popularity_dict, k):
    serendipity_scores = []
    max_rating = 5.0  # Maximum possible average rating based on our data

    for item_id in recommended_items[:k]:
        relevance = 1 if item_id in ground_truth_relevant_items else 0

        # Get popularity score, default to 0 if not found (treated as very low popularity)
        popularity_score = item_popularity_dict.get(item_id, 0)

        # Normalize popularity score to act as 'expectedness'
        # A higher popularity score means more expected, thus lower unexpectedness
        normalized_expectedness = popularity_score / max_rating if max_rating > 0 else 0

        # Unexpectedness: 1 for least popular, 0 for most popular
        unexpectedness = 1 - normalized_expectedness

        # Serendipity = relevance * unexpectedness
        item_serendipity = relevance * unexpectedness
        serendipity_scores.append(item_serendipity)

    return np.mean(serendipity_scores) if serendipity_scores else 0

In [ ]:
def evaluate_and_save_models(all_ratings_df, dataset_label, item_popularity_dict):
    print(f"\n### Starting evaluation for '{dataset_label}' dataset (using provided test file and deriving train from full history) ###\n")

    # Load global test ratings from the provided file
    test_filename = f'/content/test_{dataset_label}.csv' # This correctly points to /content/test_small.csv or /content/test_big.csv

    try:
        global_test_ratings = pd.read_csv(test_filename)
        print(f"Global Test ratings data loaded from {test_filename}")
    except FileNotFoundError:
        print(f"Error: Could not find '{test_filename}'. Please ensure this file exists in the current directory.")
        return pd.DataFrame() # Return empty DataFrame or handle error as appropriate

    print(f"Total ratings in original dataframe (all_ratings_df): {len(all_ratings_df):,}")
    print(f"Loaded Global Test ratings (for identifying eligible users): {len(global_test_ratings):,}")

    # 2. Identify users for evaluation from the GLOBAL test set
    # These are users who have activity in the most recent 'split_percentage' of the data
    test_user_counts_global = global_test_ratings['userId'].value_counts()
    eligible_users_global = test_user_counts_global[test_user_counts_global >= 3].index.tolist() # Users with >=3 ratings in the global test period
    print(f"Unique users found in global test set with >= 3 total ratings: {len(eligible_users_global)}")

    K_VALUES = [5, 10, 15]
    results = {
        'Model': [], 'k': [], 'Precision': [], 'Recall': [], 'nDCG': [], 'Serendipity': []
    }

    # Initialize lists to store metrics for each model and k
    user_metrics = {
        'Popular_Ratings': {k: {'ndcg': [], 'precision': [], 'recall': [], 'serendipity': []} for k in K_VALUES},
        'TMDB_Popular': {k: {'ndcg': [], 'precision': [], 'recall': [], 'serendipity': []} for k in K_VALUES},
        'Random': {k: {'ndcg': [], 'precision': [], 'recall': [], 'serendipity': []} for k in K_VALUES}
    }

    processed_users_count = 0
    # Iterate over users who are active in the global test period
    for user_id in eligible_users_global:
        # Get ALL ratings for this specific user from the full dataset
        user_all_history = all_ratings_df[all_ratings_df['userId'] == user_id].sort_values(by='timestamp').reset_index(drop=True)

        # Perform 80/20 chronological split for THIS USER'S HISTORY
        # This is where the user-specific train/test split happens for evaluation
        user_split_point = int(len(user_all_history) * 0.8)
        user_train_individual = user_all_history.iloc[:user_split_point]
        user_test_individual = user_all_history.iloc[user_split_point:]

        # Ensure individual train set is not empty (user must have some history to learn from)
        if user_train_individual.empty:
            continue

        known_movie_ids = set(user_train_individual['tmdbId']) # Movies known to the model for this user
        user_avg_rating = user_train_individual['rating'].mean()

        # Define 'relevant' items for ground truth from the INDIVIDUAL user's test set
        # Rating >= 4 AND Rating >= user's average rating from their train set
        ground_truth_relevant_items = user_test_individual[
            (user_test_individual['rating'] >= 4) &
            (user_test_individual['rating'] >= user_avg_rating)
        ]['tmdbId'].tolist()

        # Only evaluate user if their individual test set has at least 3 relevant items
        if not ground_truth_relevant_items or len(ground_truth_relevant_items) < 3:
            continue

        processed_users_count += 1

        # Generate recommendations using known_movie_ids from user_train_individual
        max_k = max(K_VALUES)
        recs_popular_ratings = recommend_popular_movies(user_id, max_k, known_movie_ids)
        recs_tmdb_popular = recommend_tmdb_popular_movies(user_id, max_k, known_movie_ids)
        recs_random = recommend_random_eligible_movies(user_id, max_k, known_movie_ids, eligible_random_movie_ids)

        # Calculate metrics for each model and k
        for k in K_VALUES:
            # Popular Ratings Model
            if recs_popular_ratings:
                ndcg = calculate_ndcg(recs_popular_ratings, ground_truth_relevant_items, k)
                precision, recall = calculate_precision_recall(recs_popular_ratings, ground_truth_relevant_items, k)
                serendipity = calculate_serendipity(recs_popular_ratings, ground_truth_relevant_items, item_popularity_dict, k)
                user_metrics['Popular_Ratings'][k]['ndcg'].append(ndcg)
                user_metrics['Popular_Ratings'][k]['precision'].append(precision)
                user_metrics['Popular_Ratings'][k]['recall'].append(recall)
                user_metrics['Popular_Ratings'][k]['serendipity'].append(serendipity)

            # TMDB Popular Model
            if recs_tmdb_popular:
                ndcg = calculate_ndcg(recs_tmdb_popular, ground_truth_relevant_items, k)
                precision, recall = calculate_precision_recall(recs_tmdb_popular, ground_truth_relevant_items, k)
                serendipity = calculate_serendipity(recs_tmdb_popular, ground_truth_relevant_items, item_popularity_dict, k)
                user_metrics['TMDB_Popular'][k]['ndcg'].append(ndcg)
                user_metrics['TMDB_Popular'][k]['precision'].append(precision)
                user_metrics['TMDB_Popular'][k]['recall'].append(recall)
                user_metrics['TMDB_Popular'][k]['serendipity'].append(serendipity)

            # Random Model
            if recs_random:
                ndcg = calculate_ndcg(recs_random, ground_truth_relevant_items, k)
                precision, recall = calculate_precision_recall(recs_random, ground_truth_relevant_items, k)
                serendipity = calculate_serendipity(recs_random, ground_truth_relevant_items, item_popularity_dict, k)
                user_metrics['Random'][k]['ndcg'].append(ndcg)
                user_metrics['Random'][k]['precision'].append(precision)
                user_metrics['Random'][k]['recall'].append(recall)
                user_metrics['Random'][k]['serendipity'].append(serendipity)

    print(f"Processed {processed_users_count} users for evaluation.")

    # Aggregate and store average metrics
    for model_name, metrics_by_k in user_metrics.items():
        for k in K_VALUES:
            avg_ndcg = np.mean(metrics_by_k[k]['ndcg']) if metrics_by_k[k]['ndcg'] else 0
            avg_precision = np.mean(metrics_by_k[k]['precision']) if metrics_by_k[k]['precision'] else 0
            avg_recall = np.mean(metrics_by_k[k]['recall']) if metrics_by_k[k]['recall'] else 0
            avg_serendipity = np.mean(metrics_by_k[k]['serendipity']) if metrics_by_k[k]['serendipity'] else 0

            results['Model'].append(model_name)
            results['k'].append(k)
            results['Precision'].append(avg_precision)
            results['Recall'].append(avg_recall)
            results['nDCG'].append(avg_ndcg)
            results['Serendipity'].append(avg_serendipity)

    results_df = pd.DataFrame(results)
    results_df_pivot = results_df.pivot_table(index='Model', columns='k', values=['Precision', 'Recall', 'nDCG', 'Serendipity'])
    print(f"\nResults for {dataset_label} dataset:")
    display(results_df_pivot)

    output_filename = f'baseline_metrics_{dataset_label}.csv'
    results_df_pivot.to_csv(output_filename)
    print(f"Results (metrics) saved to {output_filename}")
    return results_df_pivot

## Evaluation on Small Dataset (0.02% Test Split)

In [ ]:
# Create item_popularity_dict for Serendipity calculation
item_popularity_dict = popular_movies.set_index('tmdbId')['popularity_score'].to_dict()

small_dataset_results = evaluate_and_save_models(ratings, dataset_label='small', item_popularity_dict=item_popularity_dict)


### Starting evaluation for 'small' dataset (using provided test file and deriving train from full history) ###

Global Test ratings data loaded from /content/test_small.csv
Total ratings in original dataframe (all_ratings_df): 25,929,633
Loaded Global Test ratings (for identifying eligible users): 25,930
Unique users found in global test set with >= 3 total ratings: 582
Processed 501 users for evaluation.

Results for small dataset:


Precision                        Recall                      \
k                      5         10        15        5         10        15   
Model                                                                         
Popular_Ratings  0.064671  0.050100  0.043912  0.010567  0.014049  0.017320   
Random           0.002794  0.002595  0.002661  0.000262  0.000461  0.000696   
TMDB_Popular     0.049102  0.048303  0.045509  0.007558  0.015155  0.021320   

                Serendipity                          nDCG                      
k                        5         10        15        5         10        15  
Model                                                                          
Popular_Ratings    0.009512  0.007655  0.006868  0.074550  0.062166  0.056709  
Random             0.001893  0.001829  0.001946  0.003023  0.002823  0.002838  
TMDB_Popular       0.026039  0.020254  0.018863  0.041074  0.044049  0.044653

Results (metrics) saved to baseline_metrics_small.csv


## Evaluation on Big Dataset (0.2% Test Split)

In [ ]:
# Create item_popularity_dict for Serendipity calculation
item_popularity_dict = popular_movies.set_index('tmdbId')['popularity_score'].to_dict()

big_dataset_results = evaluate_and_save_models(ratings, dataset_label='big', item_popularity_dict=item_popularity_dict)


### Starting evaluation for 'big' dataset (using provided test file and deriving train from full history) ###

Global Test ratings data loaded from /content/test_big.csv
Total ratings in original dataframe (all_ratings_df): 25,929,633
Loaded Global Test ratings (for identifying eligible users): 518,593
Unique users found in global test set with >= 3 total ratings: 6151
Processed 5031 users for evaluation.

Results for big dataset:


Precision                        Recall                      \
k                      5         10        15        5         10        15   
Model                                                                         
Popular_Ratings  0.051560  0.041582  0.035381  0.011277  0.017723  0.021777   
Random           0.002306  0.002842  0.002690  0.000337  0.000875  0.001297   
TMDB_Popular     0.036056  0.042636  0.039091  0.006965  0.019294  0.027299   

                Serendipity                          nDCG                      
k                        5         10        15        5         10        15  
Model                                                                          
Popular_Ratings    0.007544  0.006305  0.005475  0.058825  0.051399  0.047473  
Random             0.001189  0.001505  0.001409  0.002178  0.002626  0.002653  
TMDB_Popular       0.017790  0.015139  0.014076  0.029704  0.038238  0.039344

Results (metrics) saved to baseline_metrics_big.csv
